# CNOT from Two iSWAPs - Physical DQD Implementation

**Physical setup (`DQDsystem` - `setup.py`):**
- `DQDsystem(tc, bx, Bz, Vac0, wc, gc, photon_max, epsilon_idle)` - energies in $\mu$ eV, frequencies in MHz
- Hilbert space: $\mathcal{H}_{\rm photon} \otimes \mathcal{H}_{\rm DQD1} \otimes \mathcal{H}_{\rm DQD2}  = n_{\rm photon,max} \times 4 \times 4$ dimensions
- Time unit: **ns**; frequencies in **GHz = rad/ns**; $\hbar = 0.6582\ \mu\text{eV}\cdot\text{ns}$

**Gate mechanisms (only `epsilon` is dynamically tunable; Bz, bx, tc are fixed):**
- **iSWAP**: `iswap_H(dqd)` - constant H at epsilon=0 sweet spot (cavity-mediated dispersive exchange)
- **Rx($\theta$)**: `xrot_H(dqd, target, t_start, t_end)`- EDSR drive $V_\mathrm{ac0}\cdot\tau_z\cdot\cos(E_\sigma/\hbar\cdot t)$ -> $\sigma_x$ via $\sin\bar\varphi$
- **Rz($\theta$)** `DQDSequenceCompiler.add_zrot()` - accumulates phase, offsets next EDSR drive
- Detuning cause phase difference between two qubits. Correction is needed

In [ ]:
import numpy as np
from numpy import pi
import qutip as qt
from qutip import tensor, sigmax, sigmay, sigmaz, qeye, destroy
from qutip.qip.operations import cnot, iswap
from qutip import sprepost, to_chi, process_fidelity
import matplotlib.pyplot as plt
import sys; sys.path.insert(0, ".")

from helpers import (
    DQDsystem,
    RAD_PER_NS_PER_MICROELECTRONVOLT,
    iswap_H, xrot_H, zrot_H,
    zrot_time_ns, zrot_delta_freq_GHz,
    vacuum_eigenstates, initial_full_state,
    print_summary, plot_pulse_schedule, DQDSequenceCompiler,
    run_unitary, vacuum_subspace_unitary, two_qubit_unitary,
)
from helpers.gates import _H0_ns, _eps1_op, _eps2_op, _edsr1_op, _edsr2_op

## Initialization

In [ ]:
Pi2 = 2 * pi

dqd = DQDsystem(
    tc=40, bx=2, Bz=20, Vac0=1.0,
    wc=Pi2 * 4.862e3, gc=Pi2 * 50, photon_max=5, epsilon_idle=2000.0,
)
print_summary(dqd)

J_ns  = (dqd.g_sigma**2 / dqd.d_sigma) * 1e-3   # GHz (effective exchange)
r_s, r_t = dqd.dispersive_ratios()
print(f"\ng_sigma  = {dqd.g_sigma:.3f} MHz")
print(f"d_sigma  = {dqd.d_sigma:.3f} MHz")
print(f"J        = {J_ns*1e3:.4f} MHz")
print(f"g/Delta  = {r_s:.5f}")

### Gate Duration Calculation

In [ ]:
# Derived gate parameters
Esigma_GHz  = dqd.Esigma * RAD_PER_NS_PER_MICROELECTRONVOLT              # physical EDSR carrier [GHz]
phi_bar     = dqd.phi_bar
Omega_R     = dqd.Vac0 * np.sin(phi_bar) * RAD_PER_NS_PER_MICROELECTRONVOLT   # Rabi angular freq [GHz]
t_pulse_x90 = pi / 2 / (dqd.Vac0 * np.sin(phi_bar) * RAD_PER_NS_PER_MICROELECTRONVOLT)   # Rx(pi/2) [ns]
tg_iSWAP    = dqd.iSWAP_gate_time() * 1e3       # iSWAP duration [ns]

EPS_Z       = 100.0                              # ueV - eps for physical Rz gates
delta_f_z   = zrot_delta_freq_GHz(dqd, EPS_Z)   # GHz

print(f"Esigma (carrier): {Esigma_GHz:.4f} GHz  -> period {2*pi/Esigma_GHz:.4f} ns")
print(f"phi_bar = {np.degrees(phi_bar):.3f} deg,  sin(phi_bar) = {np.sin(phi_bar):.6f}")
print(f"Omega_R (Rabi):  {Omega_R*1e3:.4f} mrad/ns  =  {Omega_R/2/pi*1e3:.4f} MHz")
print(f"t(Rx90):      {t_pulse_x90:.4f} ns")
print(f"t(iSWAP):     {tg_iSWAP:.2f} ns")
print(f"eps_Z = {EPS_Z} ueV")
print(f"Delta_fz = {delta_f_z*1e3:.4f} MHz")

# Single-DQD eigenenergies for rotating-frame correction
H_single = dqd.tc*dqd.tx + dqd.Bz/2*dqd.sz + dqd.bx/2*dqd.sx*dqd.tz
evals, _ = H_single.eigenstates(sort='low')
E_g_ns = float(evals[0]) * RAD_PER_NS_PER_MICROELECTRONVOLT   # rad/ns
E_e_ns = float(evals[1]) * RAD_PER_NS_PER_MICROELECTRONVOLT
print(f"\nSingle-DQD: E_g = {E_g_ns:.6f} GHz,  E_e = {E_e_ns:.6f} GHz")

In [ ]:
seq = DQDSequenceCompiler(dqd)
seq.add_zrot(target=1, angle=-pi/2)   # virtual Rz(-pi/2) on Q1  [no pulse]
seq.add_xrot(target=2, angle=+pi/2)   # EDSR Rx(pi/2) on Q2      [physical]
seq.add_zrot(target=2, angle=+pi/2)   # virtual Rz(+pi/2) on Q2  [no pulse]
seq.add_iswap()                        # iSWAP #1                  [physical]
seq.add_xrot(target=1, angle=+pi/2)   # EDSR Rx(pi/2) on Q1      [physical]
seq.add_iswap()                        # iSWAP #2                  [physical]
seq.add_zrot(target=2, angle=+pi/2)   # virtual Rz(+pi/2) on Q2  [no pulse]

print(seq)
print(f"\nTotal sequence time: {seq.total_time:.4f} ns  =  {seq.total_time/1e3:.4f} us")

In [ ]:
from helpers.gates import _eps1_op, _eps2_op
import scipy.linalg

H_td_A, _ = seq.compile()
H_mat      = H_td_A[0].full()

# Unified tlist over the full sequence:
#   xrot  -> fine grid (8 pts / EDSR cycle, ~0.026 ns)
#   iswap -> same dt so ODE takes ~1 internal sub-step per tlist point
#            (coarser dt is NOT faster: total sub-step count stays ~500k / iSWAP)
period_edsr = 2 * pi / Esigma_GHz
n_rx_pts    = max(2000, int(np.ceil(t_pulse_x90 / period_edsr)) * 8)
dt_rx       = t_pulse_x90 / n_rx_pts

offsets = [0.0]
for s in seq._steps:
    offsets.append(offsets[-1] + s['duration'])

parts = []
for i, step in enumerate(seq._steps):
    t0, t1 = offsets[i], offsets[i + 1]
    n = n_rx_pts if step['type'] == 'xrot' else max(2, round((t1 - t0) / dt_rx) + 1)
    parts.append(np.linspace(t0, t1, n))

tlist_full = np.unique(np.concatenate(parts))
n_isw_pts  = round(tg_iSWAP / dt_rx) + 1
print(f"tlist: {len(tlist_full):,} pts  "
      f"({n_rx_pts} per Rx,  {n_isw_pts:,} per iSWAP,  dt={dt_rx:.4f} ns)")
print("NOTE: iSWAP windows each require ~500k ODE sub-steps -> hours of runtime")

opts     = {"nsteps": 5_000_000, "atol": 1e-7, "rtol": 1e-7, "progress_bar": "tqdm"}
U_full_A = run_unitary(H_td_A, tlist_full, options=opts)
print("Done.")

# iSWAP matrix still needed for frame correction in c07
T_iswap     = tg_iSWAP
U_iswap_mat = scipy.linalg.expm(-1j * H_mat * T_iswap)

In [ ]:
# --- Verify decomposition analytically (ideal gates, no coupling error) ---
def Rx_1q(theta): return (-.5j * theta * qt.sigmax()).expm()
def Rz_1q(theta): return (-.5j * theta * qt.sigmaz()).expm()

I = qt.qeye(2)

# Circuit applied right-to-left: first gate is rightmost
U_analytic = (
    qt.tensor(I,             Rz_1q(+pi/2)) *   # Q2: Rz(+pi/2)  [step 7 / virtual in Part A]
    iswap()                                 *   # iSWAP #2
    qt.tensor(Rx_1q(pi/2),  I            ) *   # Q1: Rx(pi/2)
    iswap()                                 *   # iSWAP #1
    qt.tensor(I,             Rz_1q(+pi/2)) *   # Q2: Rz(+pi/2)
    qt.tensor(I,             Rx_1q(pi/2) ) *   # Q2: Rx(pi/2)
    qt.tensor(Rz_1q(-pi/2), I            )     # Q1: Rz(-pi/2)
)

chi_ideal  = to_chi(sprepost(cnot(), cnot().dag()))
F_analytic = process_fidelity(
    to_chi(sprepost(U_analytic, U_analytic.dag())),
    chi_ideal)
print(f"Analytical decomposition fidelity vs CNOT: {F_analytic:.6f}")
print("(should be 1.0)")
print()
print("Analytical |U|:")
print(np.abs(U_analytic.full()).round(4))

In [ ]:
# --- Part A: Extract 4x4 qubit unitary and compute fidelity ---
#
# Frame correction strategy:
#   In the lab frame, U_full_A contains both the gate rotation AND the
#   background free evolution.  The background has eps_idle on the idle qubit
#   during each Rx window, shifting its eigenfrequency by ~1490 GHz relative
#   to eps=0.  Over 102 ns that is ~150 000 uncorrected radians - using the
#   simple diag(exp(+i E_ij * T)) formula gives completely wrong phases.
#
#   Correct approach: compute the background propagator U_free
#   (identical sequence, but with EDSR drives set to zero) via expm for
#   each constant-H segment, then extract the gate in the interaction picture:
#       U_gate = U_free_dag * U_full_A
#
# Segment Hamiltonians (no EDSR):
#   Rx_Q2 window : H_static + eps_idle * H_e1   (Q1 idled)
#   iSWAP windows: H_static
#   Rx_Q1 window : H_static + eps_idle * H_e2   (Q2 idled)

H_e1_mat = _eps1_op(dqd).full()
H_e2_mat = _eps2_op(dqd).full()
eps_idle  = dqd.epsilon_idle

H_bg_rx2 = H_mat + eps_idle * H_e1_mat   # during Q2 Rx, Q1 at eps_idle
H_bg_rx1 = H_mat + eps_idle * H_e2_mat   # during Q1 Rx, Q2 at eps_idle

U_bg_rx2 = scipy.linalg.expm(-1j * H_bg_rx2 * t_pulse_x90)
U_bg_rx1 = scipy.linalg.expm(-1j * H_bg_rx1 * t_pulse_x90)

# Compose free propagator in time order (rightmost = first applied):
# rx2 -> iswap1 -> rx1 -> iswap2
U_free_mat = U_iswap_mat @ U_bg_rx1 @ U_iswap_mat @ U_bg_rx2
U_free_Q   = qt.Qobj(U_free_mat, dims=H_td_A[0].dims)

# Interaction-picture gate
U_gate_full = U_free_Q.dag() * U_full_A
U_4x4_A     = two_qubit_unitary(U_gate_full, dqd)
U_corr_A    = qt.Qobj(U_4x4_A, dims=[[2, 2], [2, 2]])

F_A = process_fidelity(to_chi(sprepost(U_corr_A, U_corr_A.dag())), chi_ideal)
print(f"Part A (DQDSequenceCompiler, virtual Z):  F = {F_A:.6f}")
print("|U_corr_A|:")
print(np.abs(U_corr_A.full()).round(4))

In [ ]:
# --- Part A: DQDSequenceCompiler pulse schedule (DC eps, EDSR envelope, g_sigma) ---
from helpers import plot_pulse_schedule

fig = plot_pulse_schedule(
    dqd, seq,
    n_pts=3000,
    title="Part A CNOT - DQDSequenceCompiler (virtual Z, EDSR + iSWAP only)",
)
plt.show()

## Single-Gate Verification (circuit.py)

`circuit.py` provides per-gate Hamiltonian builders for standalone verification.
All functions take `dqd: DQDsystem` as first argument.

| Function | Returns | Purpose |
|----------|---------|---------|
| `iswap_H(dqd)` | `[H_const]` | iSWAP Hamiltonian at eps=0 |
| `xrot_H(dqd, target, t_start, t_end)` | `[H0, [EDSR, f(t)], [H_idle, g(t)]]` | EDSR Rx gate |
| `zrot_H(dqd, target, t_start, t_end, eps_amp)` | `[H0, [H_eps, f(t)], [H_idle, g(t)]]` | DC eps Rz gate |
| `initial_full_state(dqd, i, j)` | ket | $\|0\rangle_\text{photon} \otimes \|\text{eig}_i\rangle_\text{DQD1} \otimes \|\text{eig}_j\rangle_\text{DQD2}$ |

In [ ]:
from helpers import run_state, dqd_populations

# iSWAP gate (constant H at eps=0 sweet spot)
H_iswap_full = iswap_H(dqd)
psi0         = initial_full_state(dqd, dqd1_eigenstate_idx=0, dqd2_eigenstate_idx=1)
tlist_iswap  = np.linspace(0, tg_iSWAP, 2001)

print(f"iSWAP Hilbert space: {H_iswap_full[0].shape}")
print(f"iSWAP gate time:     {tg_iSWAP:.2f} ns")
print(f"Initial state dims:  {psi0.dims}")

# EDSR Rx(pi/2) on DQD1
H_x1_full = xrot_H(dqd, target=1, t_start=0.0, t_end=t_pulse_x90)
print(f"\nRx(90 deg) terms: {len(H_x1_full)}")
print(f"  [0] H0 static:    {H_x1_full[0].shape}")
print(f"  [1] EDSR tau_z1:  {H_x1_full[1][0].shape}")
print(f"  [2] eps_idle op:  {H_x1_full[2][0].shape}")
print(f"  Rx(90 deg) gate time: {t_pulse_x90:.4f} ns")

# DC eps Rz(pi/2) on DQD1
H_z1_full = zrot_H(dqd, target=1, t_start=0.0, t_end=t_sq_z, eps_amp=EPS_Z)
print(f"\nRz(90 deg): eps={EPS_Z} ueV -> Delta_f={zrot_delta_freq_GHz(dqd, EPS_Z)*1e3:.4f} MHz -> t={t_sq_z:.4f} ns")
print(f"  Rz(90 deg) terms: {len(H_z1_full)}")

print("\nTo run single-gate simulations:")
print("  U     = run_unitary(H_iswap_full, np.linspace(0, tg_iSWAP, 2001))")
print("  U_vac = vacuum_subspace_unitary(U)")
print("  U_4x4 = two_qubit_unitary(U, dqd)")

## Bell-State Demo From The Gate Primitives

A compact entangling sequence is:
1. start from the vacuum-sector qubit ground state `|gg>`
2. apply `Rx(pi)` on Q2 to prepare approximately `|ge>`
3. hold the resonator-mediated exchange for a calibrated fraction of the iSWAP time

The second step is short enough to simulate directly with `xrot_H(...)`. The exchange step is constant in time, so we can evolve it exactly with a matrix exponential in the same `rad/ns` units used by the notebook.

In [ ]:
def project_qubit_state(psi, dqd):
    g, e = vacuum_eigenstates(dqd)[:2]
    qbasis = [
        qt.tensor(qt.basis(dqd.photon_max, 0), g, g),
        qt.tensor(qt.basis(dqd.photon_max, 0), g, e),
        qt.tensor(qt.basis(dqd.photon_max, 0), e, g),
        qt.tensor(qt.basis(dqd.photon_max, 0), e, e),
    ]
    amps = np.array([ket.overlap(psi) for ket in qbasis], dtype=complex)
    weight = float(np.vdot(amps, amps).real)
    return amps, weight


def pure_state_concurrence(amps):
    amps = amps / np.linalg.norm(amps)
    sy = np.array([[0, -1j], [1j, 0]], dtype=complex)
    yy = np.kron(sy, sy)
    return float(abs(amps.T @ yy @ amps))


def constant_h_solver(H_ns):
    evals, evecs = H_ns.eigenstates()
    basis_matrix = np.column_stack([vec.full().ravel() for vec in evecs])

    def coeffs(psi0):
        return basis_matrix.conj().T @ psi0.full().ravel()

    def evolve(state_coeffs, t_ns, dims):
        phases = np.exp(-1j * evals * t_ns)
        state_vec = basis_matrix @ (state_coeffs * phases)
        return qt.Qobj(state_vec[:, None], dims=dims)

    return coeffs, evolve


g, e = vacuum_eigenstates(dqd)[:2]
psi_gg = qt.tensor(qt.basis(dqd.photon_max, 0), g, g)
H_exchange_ns = _H0_ns(dqd)
exchange_coeffs, exchange_evolve = constant_h_solver(H_exchange_ns)
bell_target = np.array([0.0, 1.0, 1.0j, 0.0], dtype=complex) / np.sqrt(2.0)

t_x180_nom = np.pi / (dqd.Vac0 * np.sin(dqd.phi_bar) * RAD_PER_NS_PER_MICROELECTRONVOLT)
t_iswap_ns = dqd.iSWAP_gate_time() * 1e3

scan_x_ns = np.linspace(0.95 * t_x180_nom, 1.02 * t_x180_nom, 15)
scan_hold_ns = np.linspace(0.45 * t_iswap_ns, 0.55 * t_iswap_ns, 19)

best = None
for t_x_ns in scan_x_ns:
    H_x_q2 = xrot_H(dqd, target=2, t_start=0.0, t_end=t_x_ns)
    x_result = run_state(H_x_q2, psi_gg, np.linspace(0.0, t_x_ns, 1201))
    psi_after_x = x_result.states[-1]
    coeffs_after_x = exchange_coeffs(psi_after_x)
    for t_hold_ns in scan_hold_ns:
        psi_final = exchange_evolve(coeffs_after_x, t_hold_ns, psi_after_x.dims)
        amps, weight = project_qubit_state(psi_final, dqd)
        amps_norm = amps / np.linalg.norm(amps)
        concurrence = pure_state_concurrence(amps)
        bell_overlap = float(abs(np.vdot(bell_target, amps_norm)) ** 2)
        score = (bell_overlap, concurrence, weight)
        if best is None or score > best[0]:
            best = (score, t_x_ns, t_hold_ns, psi_after_x, psi_final, amps_norm, weight)

best_score, best_t_x_ns, best_t_hold_ns, psi_after_x, psi_bell, bell_amps, qubit_weight = best
best_bell_overlap, best_concurrence, best_weight = best_score

exchange_tlist_ns = np.linspace(0.0, best_t_hold_ns, 1601)
coeffs_after_x = exchange_coeffs(psi_after_x)
exchange_states = [exchange_evolve(coeffs_after_x, t_ns, psi_after_x.dims) for t_ns in exchange_tlist_ns]
exchange_basis_pops = np.array([
    np.abs(project_qubit_state(state, dqd)[0]) ** 2
    for state in exchange_states
])

labels = ["gg", "ge", "eg", "ee"]
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for idx, label in enumerate(labels):
    axes[0].plot(exchange_tlist_ns, exchange_basis_pops[:, idx], label=label)
axes[0].axvline(best_t_hold_ns, color="k", linestyle="--", linewidth=1.0)
axes[0].set_xlabel("exchange hold (ns)")
axes[0].set_ylabel("projected population")
axes[0].set_title("Projected two-qubit populations")
axes[0].legend()

axes[1].bar(labels, np.abs(bell_amps) ** 2)
axes[1].set_ylim(0.0, 1.0)
axes[1].set_ylabel("population")
axes[1].set_title("Final Bell-state populations")

plt.tight_layout()

bell_summary = {
    "Rx_pi_nominal_ns": float(t_x180_nom),
    "Rx_pi_calibrated_ns": float(best_t_x_ns),
    "exchange_hold_ns": float(best_t_hold_ns),
    "exchange_hold_over_iSWAP": float(best_t_hold_ns / t_iswap_ns),
    "Bell_overlap": float(best_bell_overlap),
    "concurrence": float(best_concurrence),
    "qubit_subspace_weight": float(best_weight),
    "leakage": float(1.0 - best_weight),
}
bell_summary